# 06 — Final Pipeline, Documentation & Delivery
**Home Credit Default Risk — Capstone Step 6**

Objective: package the entire credit scoring workflow into a production-ready Scikit-learn Pipeline that takes raw data as input and produces predictions as output.

In [1]:
import pandas as pd
import numpy as np
import joblib
import hashlib
import os
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.utils.class_weight import compute_sample_weight

DATA_DIR = '../data/'


## 1. Build the Production Pipeline

Loads **raw** `application_train.csv` (not the preprocessed CSV), with only the minimal `DAYS_EMPLOYED` sentinel fix — everything else (imputation, scaling, encoding) happens inside the pipeline itself.

In [2]:
raw = pd.read_csv(DATA_DIR + 'application_train.csv')

# Minimal transform only, per the brief
raw['DAYS_EMPLOYED'] = raw['DAYS_EMPLOYED'].replace(365243, np.nan)

X_raw = raw.drop(columns=['TARGET', 'SK_ID_CURR'])
y_raw = raw['TARGET']

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y_raw, test_size=0.2, stratify=y_raw, random_state=42
)

numeric_features = X_raw.select_dtypes(include='number').columns.tolist()
categorical_features = X_raw.select_dtypes(include='object').columns.tolist()

print(f"{len(numeric_features)} numeric, {len(categorical_features)} categorical features")


104 numeric, 16 categorical features


In [3]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

production_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', HistGradientBoostingClassifier(
        max_iter=200, max_depth=6, learning_rate=0.05,
        l2_regularization=1.0, class_weight='balanced', random_state=42
    )),
])

print("Pipeline built:")
print(production_pipeline)


Pipeline built:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['CNT_CHILDREN',
                                                   'AMT_INCOME_TOTAL',
                                                   'AMT_CREDIT', 'AMT_ANNUITY',
                                                   'AMT_GOODS_PRICE',
                                                   'REGION_POPULATION_RELATIVE',
                                                   'DAYS_BIRTH',
                                                   'DAYS_EMPLOYED',
                                               

**Rationale:** `SimpleImputer(median)` + `StandardScaler` for numeric features and `SimpleImputer(most_frequent)` + `OneHotEncoder(handle_unknown='ignore')` for categoricals.  

The classifier uses the same hyperparameters found optimal in Step 3 (`max_iter=200, max_depth=6, learning_rate=0.05, l2_regularization=1.0`). `handle_unknown='ignore'` ensures the pipeline doesn't crash on categorical values not seen during training, important for production robustness.

## 2. Fit on Full Raw Training Data

In [4]:
production_pipeline.fit(X_train_raw, y_train_raw)
print("Production pipeline trained on raw data.")

Production pipeline trained on raw data.


**Note on class balancing:** `HistGradientBoostingClassifier`'s `class_weight='balanced'` parameter is set at construction time and applied automatically during `.fit()` — no separate `sample_weight` argument is needed here (unlike the plain `GradientBoostingClassifier` used earlier in the project, which required manually computing `sample_weight`).

## 3. Evaluate on the Test Set

In [ ]:
fitted_imputer = production_pipeline.named_steps['preprocessor'].named_transformers_['num'].named_steps['imputer']
print(fitted_imputer.statistics_)

[ 0.00000000e+00  1.47600000e+05  5.14777500e+05  2.49030000e+04
  4.50000000e+05  1.88500000e-02 -1.57440000e+04 -1.64800000e+03
 -4.50500000e+03 -3.25500000e+03  9.00000000e+00  1.00000000e+00
  1.00000000e+00  0.00000000e+00  1.00000000e+00  0.00000000e+00
  0.00000000e+00  2.00000000e+00  2.00000000e+00  2.00000000e+00
  1.20000000e+01  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  5.05445533e-01
  5.65913080e-01  5.35276250e-01  8.76000000e-02  7.64000000e-02
  9.81600000e-01  7.55200000e-01  2.11000000e-02  0.00000000e+00
  1.37900000e-01  1.66700000e-01  2.08300000e-01  4.82000000e-02
  7.56000000e-02  7.44000000e-02  0.00000000e+00  3.60000000e-03
  8.40000000e-02  7.46000000e-02  9.81600000e-01  7.64800000e-01
  1.91000000e-02  0.00000000e+00  1.37900000e-01  1.66700000e-01
  2.08300000e-01  4.59000000e-02  7.71000000e-02  7.31000000e-02
  0.00000000e+00  1.10000000e-03  8.74000000e-02  7.59000000e-02
  9.81600000e-01  7.58500

In [6]:
def ks_statistic(y_true, y_proba):
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    return np.max(np.abs(tpr - fpr))

pipeline_proba = production_pipeline.predict_proba(X_test_raw)[:, 1]
pipeline_auroc = roc_auc_score(y_test_raw, pipeline_proba)
pipeline_gini = 2 * pipeline_auroc - 1
pipeline_ks = ks_statistic(y_test_raw, pipeline_proba)

print(f"Production pipeline (raw data, minimal transforms):")
print(f"  AUROC: {pipeline_auroc:.4f}")
print(f"  Gini:  {pipeline_gini:.4f}")
print(f"  KS:    {pipeline_ks:.4f}")
print(f"\nFor comparison, the fully feature-engineered model (Step 3/5): AUROC = 0.7678")

Production pipeline (raw data, minimal transforms):
  AUROC: 0.7584
  Gini:  0.5168
  KS:    0.3867

For comparison, the fully feature-engineered model (Step 3/5): AUROC = 0.7678


**Interpretation:**

The production pipeline achieves an AUROC of 0.7584 on raw, minimally-transformed data, a gap of only 0.0094 compared to the fully feature-engineered model (0.7678). This modest gap confirms the trade-off is worthwhile: the production pipeline sacrifices under 1 percentage point of AUROC in exchange for guaranteed end-to-end reproducibility, zero data leakage risk, and the ability to accept genuinely raw applicant data without any manual preprocessing. Gini (0.5168) and KS (0.3867) remain strong and well within acceptable ranges for a consumer credit model.

## 4. Save the Production Pipeline

In [7]:
os.makedirs('../models', exist_ok=True)
joblib.dump(production_pipeline, '../models/credit_scoring_pipeline.pkl')
print("Saved: ../models/credit_scoring_pipeline.pkl")


Saved: ../models/credit_scoring_pipeline.pkl


## 5. Sanity Check — Reload and Predict on Raw Rows

In [8]:
reloaded_pipeline = joblib.load('../models/credit_scoring_pipeline.pkl')

sample_raw_rows = X_test_raw.sample(n=5, random_state=42)
sample_predictions = reloaded_pipeline.predict_proba(sample_raw_rows)[:, 1]

print("Sample predictions on 5 raw, unprocessed application rows:")
for i, (idx, pred) in enumerate(zip(sample_raw_rows.index, sample_predictions), 1):
    actual = y_test_raw.loc[idx]
    print(f"  Applicant {i} (index={idx}): predicted PD = {pred:.4f}, actual TARGET = {actual}")

Sample predictions on 5 raw, unprocessed application rows:
  Applicant 1 (index=14343): predicted PD = 0.4965, actual TARGET = 0
  Applicant 2 (index=94529): predicted PD = 0.3739, actual TARGET = 0
  Applicant 3 (index=26833): predicted PD = 0.6339, actual TARGET = 0
  Applicant 4 (index=101924): predicted PD = 0.8226, actual TARGET = 1
  Applicant 5 (index=281182): predicted PD = 0.8220, actual TARGET = 0


In [9]:
print(X_test_raw.loc[101924, ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']])

EXT_SOURCE_1         NaN
EXT_SOURCE_2    0.107647
EXT_SOURCE_3    0.108226
Name: 101924, dtype: object


**Interpretation:**

The reloaded pipeline successfully produces varied predictions on raw, unprocessed applicant rows, confirming it works end-to-end without manual preprocessing. Applicant 4 (index=101924, PD=0.823) correctly identifies an actual defaulter (TARGET=1) with high confidence, both EXT_SOURCE_2 and EXT_SOURCE_3 are very low (0.108, 0.108) for this applicant, providing a strong, unambiguous risk signal despite a missing EXT_SOURCE_1. This suggests the pipeline's lack of a combined EXT_SOURCE_MEAN feature (identified in Step 5's SHAP analysis as the dominant predictor) mainly affects calibration for borderline cases with moderate or missing individual scores, rather than clear-cut cases where the remaining scores already agree strongly. This is a known trade-off documented as a limitation: the production pipeline guarantees raw-data compatibility and zero leakage at the cost of some individual-level precision compared to the fully feature-engineered model (AUROC 0.7584 vs. 0.7678). Future improvement: add EXT_SOURCE combination logic as a custom pipeline transformer step.

## 6. Version & Reproducibility Metadata

In [10]:
def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()[:16]

data_hash = file_hash(DATA_DIR + 'application_train.csv')
model_hash = file_hash('../models/credit_scoring_pipeline.pkl')

print(f"application_train.csv hash: {data_hash}")
print(f"credit_scoring_pipeline.pkl hash: {model_hash}")
print(f"\nRecord these hashes in reports/model_documentation.md for reproducibility tracking.")


application_train.csv hash: 52e96b895b1112e1
credit_scoring_pipeline.pkl hash: a23248ca8a798f0c

Record these hashes in reports/model_documentation.md for reproducibility tracking.
